In [2]:
pip install torch numpy pandas scikit-learn

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [4]:
data = fetch_california_housing()
X = data.data
y = data.target.reshape(-1, 1)

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X = scaler_X.fit_transform(X)
y = scaler_y.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
# Convert to torch tensors
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

In [6]:
# Linear Regression Model
# -----------------------------
class LinearRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

In [7]:
# Split Data Across Clients
# -----------------------------
def split_clients(X, y, num_clients=5):
    client_data = []
    size = len(X) // num_clients

    for i in range(num_clients):
        start = i * size
        end = (i + 1) * size if i != num_clients - 1 else len(X)
        client_data.append((X[start:end], y[start:end]))

    return client_data

clients = split_clients(X_train, y_train, num_clients=5)

In [8]:
# Local Training Function
# -----------------------------
def train_local(model, X, y, epochs=5, lr=0.01):
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

    return model.state_dict()

In [9]:
# Federated Averaging (FedAvg)
# -----------------------------
def fed_avg(global_model, client_weights):
    avg_weights = {}

    for key in client_weights[0].keys():
        avg_weights[key] = torch.mean(
            torch.stack([client_weights[i][key] for i in range(len(client_weights))]),
            dim=0
        )

    global_model.load_state_dict(avg_weights)
    return global_model

In [10]:
# Federated Training Loop
# -----------------------------
input_dim = X_train.shape[1]
global_model = LinearRegressionModel(input_dim)

rounds = 10

for r in range(rounds):
    client_weights = []

    for client_X, client_y in clients:
        local_model = LinearRegressionModel(input_dim)
        local_model.load_state_dict(global_model.state_dict())

        weights = train_local(local_model, client_X, client_y)
        client_weights.append(weights)

    global_model = fed_avg(global_model, client_weights)
    print(f"Round {r+1} completed")

Round 1 completed
Round 2 completed
Round 3 completed
Round 4 completed
Round 5 completed
Round 6 completed
Round 7 completed
Round 8 completed
Round 9 completed
Round 10 completed


In [11]:
# Evaluate Global Model
# -----------------------------
global_model.eval()
with torch.no_grad():
    predictions = global_model(X_test)
    mse = nn.MSELoss()(predictions, y_test)

print("Final Test MSE:", mse.item())

Final Test MSE: 0.6124114990234375
